In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import *

1. Data preparation

In [ ]:
# Load fundamental-model R-wave dispersion data
data = np.loadtxt("examples/example_01.txt")
f = data[:,0]
vr = data[:,1]
vr_std = data[:,2] # you can specify the standard deviation yourself if not provided

# Sort the dispersion data from smallest to largest frequency
sorted_indices = np.argsort(f)
f = f[sorted_indices]
vr = vr[sorted_indices]
vr_std = vr_std[sorted_indices]

# Visualize the dispersion data
plt.errorbar(f, vr, yerr=vr_std, fmt='o', markersize=3, capsize=5)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Phase Velocity (m/s)')
plt.xscale('log')

2. Dispersion curve normalization

In [ ]:
# Estimate halfspace depth and S-wave velocity bounds
vs_bounds_halfspace = np.array([1.05*np.max(vr),3*np.max(vr)])
depth_bounds_halfspace = np.array([0.2, 0.7])*np.max(vr/f)
print('vs_bounds_halfspace (m/s):', vs_bounds_halfspace)
print('depth_bounds_halfspace (m):', depth_bounds_halfspace)


# Normalization and resampling with parallel processing
depth, vs_hs, f_vr_resampled = scale_and_resample_dc(f, vr, vs_bounds_halfspace, depth_bounds_halfspace, 
                                                          nd = 200, nv = 200, v_scale='log', d_scale='log')
f_vr_resampled = np.array(f_vr_resampled, dtype=object)

3. Inversion

In [ ]:
model_path = "models/PADIT_monotonic.pth"  # Replace with "models/PADIT_LVZ_inclusive.pth" to use the LVZ-inclusive model.
vs_predict_no_hs = run_prediction(model_path, f_vr_resampled)

4. Denormalization

In [ ]:
# Append the normalized half-space velocity (Vs / Vs_hs = 1).
# The half-space thickness is represented as zero during denormalization.
vs_predict = np.hstack((vs_predict_no_hs, np.full((vs_predict_no_hs.shape[0], 1), 1)))
thickness_inverted, vs_inverted = denormalize_dc(vs_predict, depth, vs_hs)

5. Forward computation

In [ ]:
f_scaled = depth.reshape(-1, 1) /vs_hs.reshape(-1, 1) * f.reshape(1, -1)
vr_inverted = forward_parallel(vs_predict, f_scaled, n_jobs=-1)
vr_inverted = vr_inverted * vs_hs.reshape(-1, 1)

6. Evaluate

In [ ]:
qualified_indices, vr_misfit = find_qualified_indices_and_rank(vr, vr_std, vr_inverted,limit=1)
print(f"Number of qualified models: {len(qualified_indices)}")
thickness_qualified = thickness_inverted[qualified_indices]
vs_qualified = vs_inverted[qualified_indices]
vr_qualified = vr_inverted[qualified_indices]

7. Plot

In [ ]:
# Select the number of models to plot, starting from the best-ranked model
plot_number = min(1000, len(vr_qualified))

fig, axs = plt.subplots(1, 2, 
                        figsize=(8, 4), 
                        gridspec_kw={'width_ratios': [1.8, 1]})

for i in range(plot_number):
    axs[0].plot(f, vr_qualified[i], 'b-')
axs[0].errorbar(f,vr, yerr=vr_std.reshape(-1), fmt='o', ecolor='red', elinewidth=1, markersize=3,capsize=5,color = 'red')
axs[0].set_xscale('log')
axs[0].set_xlabel('Frequency (Hz)')
axs[0].set_ylabel('Vr (m/s)')

depth_plot = get_depth_for_plot(thickness_qualified)
vs_plot = np.repeat(vs_qualified,2,axis=1)
for i in range(plot_number):
    axs[1].plot(vs_plot[i],depth_plot[i],color = 'blue')
axs[1].set_ylim(0, np.max(depth_plot))
axs[1].set_xlim(0, np.max(vs_plot) * 1.05)
axs[1].set_xlabel('Vs (m/s)')
axs[1].set_ylabel('Depth (m)')
axs[1].invert_yaxis()
plt.tight_layout()